# Spark Profiling & Optimization - Exercises

Continuation of `compar-lab02-spark-dataframes`. In the showcase notebook
you saw four optimizations demonstrated on simple aggregations. Here you
apply the same method - **profile -> diagnose -> fix -> profile again ->
quantify** - to a real query: the *"top rated movie of every year since
2000"* pipeline that was left unfinished in lab02's `movies.ipynb`.

For every exercise: fill in the `TODO`, re-run its `with profiler.profile(...)`
block, and answer the question in the markdown cell below it **in your own
words, with numbers from your run**.

## Setup

In [ ]:
master = "spark://spark-master:7077"   # use "spark://spark-master:7077" when running inside the Docker cluster
dataFolder = "/data/lab02/" if master.startswith("spark://") else "../../compar-lab02-spark-dataframes/data/"


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType
from pyspark.sql import Window

spark_builder = (SparkSession.builder
         .master(master)
         .appName("spark-profiling-exercises")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "1536m")   # stay under the worker's --memory 2G cap
         .config("spark.ui.showConsoleProgress", "false"))

spark = spark_builder.getOrCreate()

print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default parallelism (cores seen by the driver):", spark.sparkContext.defaultParallelism)


In [ ]:
import sys
sys.path.append("/workspace/lab03")

from spark_profiler import SparkProfiler

profiler = SparkProfiler(spark)

## Step 0 - Baseline

Below is the naive version of the pipeline: schema inference, no column
pruning, default partitioning, a plain join, nothing cached. Run it exactly
as given and record the baseline numbers - every later exercise is compared
against this run.

In [ ]:
with profiler.profile("0-baseline"):
    movies = (spark.read.option("header", "true").option("inferSchema", "true")
              .csv(dataFolder + "movies.csv.gz"))
    ratings = (spark.read.option("header", "true").option("inferSchema", "true")
               .csv(dataFolder + "ratings1.csv.gz"))

    movies_year = (movies
                   .withColumn("year_str", sf.regexp_extract("title", r'^(.*) \(([0-9 \-]*)\)$', 2))
                   .withColumn("year", sf.when(sf.col("year_str").rlike(r'^[0-9]+$'), sf.col("year_str").cast(IntegerType())))
                   .drop("year_str")
                   .filter("year IS NOT NULL"))

    avg_ratings = (movies_year.join(ratings, on="movieId")
                   .groupBy("year", "title")
                   .agg(sf.round(sf.avg("rating"), 2).alias("avg_rating"),
                        sf.count("*").alias("num_ratings"))
                   .filter("num_ratings > 100"))

    w = Window.partitionBy("year").orderBy(sf.desc("avg_rating"))
    top_per_year = (avg_ratings
                    .withColumn("rank", sf.row_number().over(w))
                    .filter("rank = 1 and year >= 2000")
                    .orderBy("year")
                    .select("year", "title", "avg_rating", "num_ratings"))

    top_per_year.show(30, truncate=False)


**Q0.** How many partitions does `ratings` have right after being read
(`ratings.rdd.getNumPartitions()`)? How does that compare to the number of
cores available in your cluster (`spark.sparkContext.defaultParallelism`)?

In [ ]:
print("ratings partitions:", ratings.rdd.getNumPartitions())
print("default parallelism:", spark.sparkContext.defaultParallelism)
profiler.report(["0-baseline"])


## Exercise A - Partitioning

Fix the partition-count problem from Q0. Re-run the *entire* pipeline inside
a new `with profiler.profile("A-repartitioned"):` block, changing only how
`ratings` is read/partitioned.

In [ ]:
with profiler.profile("A-repartitioned"):
    movies = (spark.read.option("header", "true").option("inferSchema", "true")
              .csv(dataFolder + "movies.csv.gz"))
    ratings = (spark.read.option("header", "true").option("inferSchema", "true")
               .csv(dataFolder + "ratings1.csv.gz")
               # TODO: fix the partitioning problem here
               )

    movies_year = (movies
                   .withColumn("year_str", sf.regexp_extract("title", r'^(.*) \(([0-9 \-]*)\)$', 2))
                   .withColumn("year", sf.when(sf.col("year_str").rlike(r'^[0-9]+$'), sf.col("year_str").cast(IntegerType())))
                   .drop("year_str")
                   .filter("year IS NOT NULL"))

    avg_ratings = (movies_year.join(ratings, on="movieId")
                   .groupBy("year", "title")
                   .agg(sf.round(sf.avg("rating"), 2).alias("avg_rating"),
                        sf.count("*").alias("num_ratings"))
                   .filter("num_ratings > 100"))

    w = Window.partitionBy("year").orderBy(sf.desc("avg_rating"))
    top_per_year = (avg_ratings
                    .withColumn("rank", sf.row_number().over(w))
                    .filter("rank = 1 and year >= 2000")
                    .orderBy("year")
                    .select("year", "title", "avg_rating", "num_ratings"))

    top_per_year.show(30, truncate=False)

profiler.compare("0-baseline", "A-repartitioned")


**Q A.** What did you change, and why? Quote the before/after numbers from
`profiler.compare(...)` that back up your answer.

## Exercise B - Schema & column pruning

Give `ratings` an explicit schema (no `inferSchema`) and drop every column
neither table actually needs before the join. Which columns does this
pipeline use from each table?

In [ ]:
# TODO: define an explicit schema for ratings (userId, movieId, rating, timestamp)
ratings_schema = ...

with profiler.profile("B-schema-pruned"):
    movies = (spark.read.option("header", "true").option("inferSchema", "true")
              .csv(dataFolder + "movies.csv.gz"))
    ratings = (spark.read.option("header", "true").schema(ratings_schema)
               .csv(dataFolder + "ratings*.csv.gz")
               # TODO: reuse your fix from Exercise A, then keep only the
               # columns this pipeline actually needs
               )

    movies_year = (movies
                   .withColumn("year_str", sf.regexp_extract("title", r'^(.*) \(([0-9 \-]*)\)$', 2))
                   .withColumn("year", sf.when(sf.col("year_str").rlike(r'^[0-9]+$'), sf.col("year_str").cast(IntegerType())))
                   .drop("year_str")
                   .filter("year IS NOT NULL")
                   # TODO: prune movies too - which of its columns are used downstream?
                   )

    avg_ratings = (movies_year.join(ratings, on="movieId")
                   .groupBy("year", "title")
                   .agg(sf.round(sf.avg("rating"), 2).alias("avg_rating"),
                        sf.count("*").alias("num_ratings"))
                   .filter("num_ratings > 100"))

    w = Window.partitionBy("year").orderBy(sf.desc("avg_rating"))
    top_per_year = (avg_ratings
                    .withColumn("rank", sf.row_number().over(w))
                    .filter("rank = 1 and year >= 2000")
                    .orderBy("year")
                    .select("year", "title", "avg_rating", "num_ratings"))

    top_per_year.show(30, truncate=False)

profiler.compare("A-repartitioned", "B-schema-pruned")


**Q B.** Which columns did you drop, and from which table? What happened to
`shuffle_write_bytes`?

## Exercise C - Join strategy

`movies` is tiny, `ratings` is not. Use `.explain()` to check whether the
join inside `avg_ratings` is already being broadcast. Then force the
*opposite* strategy of whatever you found (`broadcast(movies_year)` or
`.hint("merge")`) so you can measure the difference yourself.

In [ ]:
with profiler.profile("C-join-check"):
    joined = movies_year.join(ratings, on="movieId")
    joined.groupBy("title").count().limit(1).collect()

joined.explain()

# TODO: now force the opposite join strategy (add broadcast(...) or .hint("merge")
# to the join above) and profile it as "C-join-forced". Which one is faster here,
# and does that match what you'd expect from the size of movies_year vs ratings?


**Q C.** Was the join already broadcast by default? Why - or why not (hint:
look at `spark.sql.autoBroadcastJoinThreshold` and the size of
`movies.csv.gz` after your Exercise B pruning)? What changed when you forced
the other strategy?

## Exercise E - Filter pushdown

Look again at where `year >= 2000` is applied in the baseline pipeline:
it's the very last filter, on `top_per_year`, *after* the join and the
window function. MovieLens spans about a century of movies, so most of
`movies_year` gets joined against `ratings` and shuffled through the
`groupBy`, only to be thrown away at the very end. Move that filter to
right after `movies_year` is computed - before the join - and profile the
result as `"E-filter-pushed"`.

In [ ]:
with profiler.profile("E-filter-pushed"):
    movies = (spark.read.option("header", "true").option("inferSchema", "true")
              .csv(dataFolder + "movies.csv.gz"))
    ratings = (spark.read.option("header", "true").schema(ratings_schema)
               .csv(dataFolder + "ratings*.csv.gz")
               # TODO: reuse your Exercise A/B fixes here
               )

    movies_year = (movies
                   .withColumn("year_str", sf.regexp_extract("title", r'^(.*) \(([0-9 \-]*)\)$', 2))
                   .withColumn("year", sf.when(sf.col("year_str").rlike(r'^[0-9]+$'), sf.col("year_str").cast(IntegerType())))
                   .drop("year_str")
                   .filter("year IS NOT NULL")
                   # TODO: push the "year >= 2000" filter here, as early as possible
                   )

    avg_ratings = (movies_year.join(ratings, on="movieId")
                   .groupBy("year", "title")
                   .agg(sf.round(sf.avg("rating"), 2).alias("avg_rating"),
                        sf.count("*").alias("num_ratings"))
                   .filter("num_ratings > 100"))

    w = Window.partitionBy("year").orderBy(sf.desc("avg_rating"))
    top_per_year = (avg_ratings
                    .withColumn("rank", sf.row_number().over(w))
                    .filter("rank = 1")   # the year filter has already been applied upstream
                    .orderBy("year")
                    .select("year", "title", "avg_rating", "num_ratings"))

    top_per_year.show(30, truncate=False)

profiler.compare("0-baseline", "E-filter-pushed")


**Q E.** Compare `top_per_year.explain()` before and after this change (or
against `C-join-check`'s plan). Does Spark's optimizer ever push a filter
like this down on its own - and if so, under what conditions can it prove
that's safe? What happened to `shuffle_read_bytes`/`shuffle_write_bytes`,
and to the row counts feeding the join?

## Exercise D - Caching a reused DataFrame

`movies_year` is recomputed from scratch by *every* action that depends on
it, even though nothing about it changes across this notebook. Identify a
DataFrame in the pipeline that is used more than once downstream, and decide
whether caching it is worth it here - think about both its size and how many
times it gets reused.

In [ ]:
# TODO: add .cache() to the DataFrame you identified, trigger it with an
# action (e.g. .count()), then re-run the full pipeline profiled as "D-cached".
# Don't forget .unpersist() once you're done experimenting with it below.


**Q D.** What did you cache, and what's the tradeoff? Would caching still
make sense if this notebook only ever called `top_per_year.show()` once and
nothing else touched `movies_year`?

## Put it all together

Combine every fix from A-E into one final pipeline and profile it as
`"final"`.

In [ ]:
with profiler.profile("final"):
    # TODO: assemble your final pipeline here, combining Exercises A-E
    pass


In [ ]:
profiler.report()


In [ ]:
profiler.compare("0-baseline", "final")


**Final summary.** In 3-4 sentences: which single change gave the biggest
improvement, which gave the smallest (or none), and why do you think that
is, given this dataset and this cluster size?

## Stretch goal (optional) - does it scale?

Everything above ran against whatever number of workers your cluster
happened to have.

1. Shut down this notebook's kernel (*Kernel -> Shut Down Kernel*) so the
   Spark application releases its executors.
2. From a terminal: `docker compose up -d --scale spark-worker=<N>` - try
   1, then 3, then 6 workers.
3. Re-run the *Setup* cells plus `0-baseline` and `final` for each worker
   count, noting the wall time for both.
4. Tabulate wall time vs worker count for `0-baseline` and for `final`.
   Which one actually gets faster as you add workers? Why?

For a version you can run without keeping a notebook kernel alive across
restarts, see `apps/run_tuned_pipeline.py` in this repo - `spark-submit` it
directly and watch the terminal / Spark UI instead.

In [ ]:
spark.stop()
